In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")
catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "456_build_gold_vw_dim_date.py"
RUN_ID = str(uuid.uuid4())

SOURCE_TABLE = f"{catalog}.gold.dim_date"
TARGET_VIEW = f"{catalog}.gold.vw_dim_date"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build gold.vw_dim_date
# ============================================================
try:
    spark.sql(f"DROP VIEW IF EXISTS {TARGET_VIEW}")

    spark.sql(f"""
    CREATE VIEW {TARGET_VIEW} AS
    WITH relative_dates AS (
        SELECT
              *
            , datediff(calendar_date, current_date()) AS relative_day
            , (
                ((year(calendar_date) - year(current_date())) * 12)
                + (month(calendar_date) - month(current_date()))
              ) AS relative_month
            , (year(calendar_date) - year(current_date())) AS relative_year
            , CASE
                  WHEN calendar_date BETWEEN trunc(current_date(), 'YEAR') AND current_date() THEN 1
                  ELSE 0
              END AS is_ytd_flag
            , CASE
                  WHEN calendar_date BETWEEN trunc(current_date(), 'QUARTER') AND current_date() THEN 1
                  ELSE 0
              END AS is_qtd_flag
            , CASE
                  WHEN calendar_date BETWEEN trunc(current_date(), 'MONTH') AND current_date() THEN 1
                  ELSE 0
              END AS is_mtd_flag
        FROM {SOURCE_TABLE}
    )
    SELECT
          calendar_date          AS `Calendar Date`
        , date_key               AS `Date Key`
        , day_name               AS `Day Name`
        , day_name_short         AS `Day Name Short`
        , day_of_month           AS `Day Of Month`
        , day_of_week_iso        AS `Day Of Week ISO`
        , quarter                AS `Quarter`
        , quarter_label          AS `Quarter Label`
        , month                  AS `Month`
        , year                   AS `Year`
        , day_of_year            AS `Day Of Year`
        , week_of_year           AS `Week Of Year`
        , first_day_of_month     AS `First Day Of Month`
        , last_day_of_month      AS `Last Day Of Month`
        , first_day_of_quarter   AS `First Day Of Quarter`
        , last_day_of_quarter    AS `Last Day Of Quarter`
        , first_day_of_year      AS `First Day Of Year`
        , last_day_of_year       AS `Last Day Of Year`
        , month_name             AS `Month Name`
        , month_name_short       AS `Month Name Short`
        , month_year             AS `Month Year`
        , period                 AS `Period`
        , year_quarter           AS `Year Quarter`
        , year_month_key         AS `Year Month Key`
        , month_sort             AS `Month Sort`
        , flag_weekday           AS `Flag Weekday`
        , flag_last_day_of_month AS `Flag Last Day Of Month`
        , fiscal_year            AS `Fiscal Year`
        , fiscal_quarter         AS `Fiscal Quarter`
        , fiscal_year_label      AS `Fiscal Year Label`
        , fiscal_quarter_label   AS `Fiscal Quarter Label`
        , is_holiday             AS `Is Holiday`
        , holiday_key            AS `Holiday Key`
        , holiday_name           AS `Holiday Name`
        , holiday_description    AS `Holiday Description`
        , holiday_country_code   AS `Holiday Country Code`
        , holiday_group          AS `Holiday Group`
        , relative_day           AS `Relative Day`
        , relative_month         AS `Relative Month`
        , relative_year          AS `Relative Year`
        , is_ytd_flag            AS `Is YTD Flag`
        , is_qtd_flag            AS `Is QTD Flag`
        , is_mtd_flag            AS `Is MTD Flag`
    FROM relative_dates
    """)

    row_count = spark.sql(f"SELECT COUNT(*) AS row_count FROM {TARGET_VIEW}").collect()[0]["row_count"]

    log_run("SUCCESS", row_count, "Built gold.vw_dim_date successfully.")

    print("=" * 70)
    print("Built gold.vw_dim_date")
    print(f"Row count: {row_count:,}")
    print("=" * 70)

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise